In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix, lil_matrix
from pyproj import Transformer
from sklearn.preprocessing import StandardScaler
import pyreadr 
from pathlib import Path
import os

os.chdir(Path.cwd().parent)


In [ ]:
# PPC: Location-level time-averaged proportion
# One-step-ahead posterior ranking p-values
# 5 MODELS:
#   IID / BYM shared / BYM weekly / BYM+cov / BYM weekly+cov
import numpy as np
import pyreadr
import pickle
import pandas as pd
from pathlib import Path
from tqdm import tqdm

np.random.seed(1)
# Paths and settings
BASE_DIR = Path(r"path/to/snow/data-and-results")
PERIOD = 52


no_nbs = np.array([
    57,170,236,269,343,685,946,947,989,
    1037,1084,1090,1109,1118,1127,1176,1203
]) - 1
# Data
snow = pyreadr.read_r(BASE_DIR / "snow_cleaned_full.Rda")
snow = list(snow.values())[0]

coords_full = snow.iloc[:, :2].to_numpy()
y_full = snow.iloc[:, 2:].to_numpy()
# Data — EXACT SAME 2PC FILTER AS MODEL FITTING
import geopandas as gpd
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

coords = coords_full
y = y_full

S, T = y.shape
print("Using FULL S =", S)
# OBSERVED STATISTIC
T_obs = y[:,1:].mean(axis=1)  # (S,)
# TIME COVARIATES
t_raw = np.arange(1, T+1)
t_trend = (t_raw - t_raw.mean()) / t_raw.std()

cov4 = np.column_stack([
    np.ones(T),
    np.cos(2*np.pi*t_raw/PERIOD),
    np.sin(2*np.pi*t_raw/PERIOD),
    t_trend
])

cov8 = np.column_stack([
    np.ones(T), np.ones(T),
    np.cos(2*np.pi*t_raw/PERIOD), np.cos(2*np.pi*t_raw/PERIOD),
    np.sin(2*np.pi*t_raw/PERIOD), np.sin(2*np.pi*t_raw/PERIOD),
    t_trend, t_trend
])

week = (t_raw - 1) % 52
# SPATIAL COVARIATES (SAFE VERSION)
# ---- latitude
lat = coords[:,1]
lat = (lat - lat.mean()) / lat.std()

# ---- elevation (safe read)
elev_df = pd.read_csv(BASE_DIR / "curr_elev.csv")
elev_all = elev_df.iloc[:,3].to_numpy()

nnbs_df = pd.read_csv(BASE_DIR / "nnbs_elev.csv", sep="\t")
nnbs_elev = nnbs_df.iloc[:,2].to_numpy()

mask = np.ones(S, dtype=bool)
mask[no_nbs] = False

elev_full = np.zeros(S)
elev_full[mask] = elev_all
elev_full[no_nbs] = nnbs_elev

elev = (elev_full - elev_full.mean()) / elev_full.std()


# ---- temperature
snow_temp = pyreadr.read_r(BASE_DIR / "snow_temp_full.Rda")
snow_temp = list(snow_temp.values())[0].reset_index(drop=True)

temp = snow_temp.iloc[:,2:].to_numpy()
temp = (temp - temp.mean()) / temp.std()
# LOAD POSTERIORS
thin = 15
n_chains = 10

iid01_list = []
iid10_list = []

for c in range(n_chains):
    d01 = np.load(BASE_DIR / f"p01_ind_all_chain{c}.npz")
    d10 = np.load(BASE_DIR / f"p10_ind_all_chain{c}.npz")

    iid01_list.append(d01["all_theta"][:, ::thin])
    iid10_list.append(d10["all_theta"][:, ::thin])

iid01 = np.concatenate(iid01_list, axis=1)
iid10 = np.concatenate(iid10_list, axis=1)


eta01_list = []
tau01_list = []
eta10_list = []
tau10_list = []

for c in range(10):

    with open(BASE_DIR / f"p01_weekly_chain{c}.pkl","rb") as f:
        d = pickle.load(f)
    eta01_list.append(d["eta"][:, ::15])
    tau01_list.append(d["tau"][:, ::15])

    with open(BASE_DIR / f"p10_weekly_chain{c}.pkl","rb") as f:
        d = pickle.load(f)
    eta10_list.append(d["eta"][:, ::15])
    tau10_list.append(d["tau"][:, ::15])

eta01_week = np.concatenate(eta01_list, axis=1)
tau01_week = np.concatenate(tau01_list, axis=1)
eta10_week = np.concatenate(eta10_list, axis=1)
tau10_week = np.concatenate(tau10_list, axis=1)


eta01_wf_list = []
tau01_wf_list = []
eta10_wf_list = []
tau10_wf_list = []

for c in range(10):

    with open(BASE_DIR / f"p01_weekly_cov_chain{c}.pkl","rb") as f:
        d = pickle.load(f)
    eta01_wf_list.append(d["eta"][:, ::15])
    tau01_wf_list.append(d["tau"][:, ::15])

    with open(BASE_DIR / f"p10_weekly_cov_chain{c}.pkl","rb") as f:
        d = pickle.load(f)
    eta10_wf_list.append(d["eta"][:, ::15])
    tau10_wf_list.append(d["tau"][:, ::15])

eta01_wf = np.concatenate(eta01_wf_list, axis=1)
tau01_wf = np.concatenate(tau01_wf_list, axis=1)
eta10_wf = np.concatenate(eta10_wf_list, axis=1)
tau10_wf = np.concatenate(tau10_wf_list, axis=1)

M = iid01.shape[1]
N_REP = M
# SIGMOID
def sigmoid(z):
    z = np.clip(z,-30,30)
    return 1/(1+np.exp(-z))
# GENERIC ONE-STEP-AHEAD SIMULATOR
def simulate(get_eta):

    draws = np.arange(M)
    T_pred = np.zeros((N_REP, S))

    for j, m in enumerate(tqdm(draws)):

        y_rep = np.zeros((S,T), dtype=int)
        y_rep[:,0] = y[:,0]

        for t in range(1,T):

            eta01, eta10 = get_eta(m,t-1)

            p01 = sigmoid(eta01)
            p10 = sigmoid(eta10)

            prev = y_rep[:,t-1]

            prob = np.where(prev==0, p01, 1-p10)

            y_rep[:,t] = np.random.binomial(1, prob)

        T_pred[j,:] = y_rep[:,1:].mean(axis=1)

    return np.mean(T_pred >= T_obs, axis=0)
# ETA DEFINITIONS
def eta_iid(m,t):
    e1 = sum(cov4[t,k] * iid01[k*S:(k+1)*S,m] for k in range(4))
    e2 = sum(cov4[t,k] * iid10[k*S:(k+1)*S,m] for k in range(4))
    return e1, e2


def eta_week(m,t):
    w = week[t]
    e1 = np.zeros(S)
    e2 = np.zeros(S)
    for k in range(8):
        beta1 = eta01_week[k*S:(k+1)*S,m]
        beta2 = eta10_week[k*S:(k+1)*S,m]
        e1 += cov8[t,k] * beta1 * tau01_week[k*52+w,m]
        e2 += cov8[t,k] * beta2 * tau10_week[k*52+w,m]
    return e1, e2

def eta_week_cov(m, t):

    w = week[t]

    e1 = np.zeros(S)
    e2 = np.zeros(S)
    # 1. spatial BYM weekly part
    for k in range(8):

        # spatial beta blocks
        beta1 = eta01_wf[k*S:(k+1)*S, m]
        beta2 = eta10_wf[k*S:(k+1)*S, m]

        # weekly scaling
        scale1 = tau01_wf[k*52 + w, m]
        scale2 = tau10_wf[k*52 + w, m]

        e1 += cov8[t, k] * beta1 * scale1
        e2 += cov8[t, k] * beta2 * scale2
    # 2. global covariate part
    gamma1 = eta01_wf[8*S : 8*S+3, m]
    gamma2 = eta10_wf[8*S : 8*S+3, m]

    fac = np.column_stack([
        t_trend[t] * lat,
        t_trend[t] * elev,
        t_trend[t] * temp[:, t]
    ])

    e1 += fac @ gamma1
    e2 += fac @ gamma2

    return e1, e2
# RUN ALL MODELS
print("\nIID")
p_iid = simulate(eta_iid)

print("\nBYM weekly")
p_week = simulate(eta_week)

print("\nBYM weekly + cov")
p_week_cov = simulate(eta_week_cov)
# SUMMARY
def summarize(name, pvals):
    print("\n", name)
    print("Mean:", pvals.mean())
    print("SD:", pvals.std())
    print("Pr(p<0.05):", np.mean(pvals < 0.05))
    print("Pr(p>0.95):", np.mean(pvals > 0.95))

summarize("IID", p_iid)
summarize("BYM weekly", p_week)
summarize("BYM weekly+cov", p_week_cov)

In [ ]:
# OBSERVED WEEKLY STATISTIC
week_indices = [(week == w) for w in range(52)]

T_obs_week = np.array([
    y[:, idx].mean() for idx in week_indices
])

print("Observed weekly shape:", T_obs_week.shape)
# LOAD POSTERIORS
iid01 = np.concatenate([
    np.load(BASE_DIR / f"p01_ind_all_chain{c}.npz")["all_theta"][:, ::thin]
    for c in range(n_chains)
], axis=1)

iid10 = np.concatenate([
    np.load(BASE_DIR / f"p10_ind_all_chain{c}.npz")["all_theta"][:, ::thin]
    for c in range(n_chains)
], axis=1)


# BYM weekly
eta01_week, tau01_week, eta10_week, tau10_week = [], [], [], []

for c in range(10):
    with open(BASE_DIR / f"p01_weekly_chain{c}.pkl","rb") as f:
        d = pickle.load(f)
    eta01_week.append(d["eta"][:, ::15])
    tau01_week.append(d["tau"][:, ::15])

    with open(BASE_DIR / f"p10_weekly_chain{c}.pkl","rb") as f:
        d = pickle.load(f)
    eta10_week.append(d["eta"][:, ::15])
    tau10_week.append(d["tau"][:, ::15])

eta01_week = np.concatenate(eta01_week, axis=1)
tau01_week = np.concatenate(tau01_week, axis=1)
eta10_week = np.concatenate(eta10_week, axis=1)
tau10_week = np.concatenate(tau10_week, axis=1)


# BYM weekly + cov
eta01_wf, tau01_wf, eta10_wf, tau10_wf = [], [], [], []

for c in range(10):
    with open(BASE_DIR / f"p01_weekly_cov_chain{c}.pkl","rb") as f:
        d = pickle.load(f)
    eta01_wf.append(d["eta"][:, ::15])
    tau01_wf.append(d["tau"][:, ::15])

    with open(BASE_DIR / f"p10_weekly_cov_chain{c}.pkl","rb") as f:
        d = pickle.load(f)
    eta10_wf.append(d["eta"][:, ::15])
    tau10_wf.append(d["tau"][:, ::15])

eta01_wf = np.concatenate(eta01_wf, axis=1)
tau01_wf = np.concatenate(tau01_wf, axis=1)
eta10_wf = np.concatenate(eta10_wf, axis=1)
tau10_wf = np.concatenate(tau10_wf, axis=1)


M = iid01.shape[1]
# SIGMOID
def sigmoid(z):
    return 1/(1+np.exp(-np.clip(z,-30,30)))
# ETA DEFINITIONS
def eta_iid(m,t):
    e1 = sum(cov4[t,k]*iid01[k*S:(k+1)*S,m] for k in range(4))
    e2 = sum(cov4[t,k]*iid10[k*S:(k+1)*S,m] for k in range(4))
    return e1,e2


def eta_week(m,t):
    w = week[t]
    e1 = np.zeros(S)
    e2 = np.zeros(S)
    for k in range(8):
        e1 += cov8[t,k]*eta01_week[k*S:(k+1)*S,m]*tau01_week[k*52+w,m]
        e2 += cov8[t,k]*eta10_week[k*S:(k+1)*S,m]*tau10_week[k*52+w,m]
    return e1,e2


def eta_week_cov(m,t):
    w = week[t]
    e1 = np.zeros(S)
    e2 = np.zeros(S)

    for k in range(8):
        e1 += cov8[t,k]*eta01_wf[k*S:(k+1)*S,m]*tau01_wf[k*52+w,m]
        e2 += cov8[t,k]*eta10_wf[k*S:(k+1)*S,m]*tau10_wf[k*52+w,m]

    gamma1 = eta01_wf[8*S:8*S+3,m]
    gamma2 = eta10_wf[8*S:8*S+3,m]

    fac = np.column_stack([
        t_trend[t]*lat,
        t_trend[t]*elev,
        t_trend[t]*temp[:,t]
    ])

    return e1 + fac @ gamma1, e2 + fac @ gamma2
# SIMULATOR
def simulate_weekly_summary(get_eta):

    T_pred = np.zeros((M, 52))

    for j, m in enumerate(tqdm(range(M))):

        y_rep = np.zeros((S,T), dtype=int)
        y_rep[:,0] = y[:,0]

        for t in range(1,T):
            eta01, eta10 = get_eta(m, t-1)
            p01 = sigmoid(eta01)
            p10 = sigmoid(eta10)
            prev = y_rep[:,t-1]
            y_rep[:,t] = np.random.binomial(1, np.where(prev==0,p01,1-p10))

        for w in range(52):
            T_pred[j,w] = y_rep[:, week_indices[w]].mean()

    return (
        T_pred.mean(axis=0),
        np.percentile(T_pred, 2.5, axis=0),
        np.percentile(T_pred, 97.5, axis=0)
    )
# RUN
print("\nIID")
mean_iid, low_iid, up_iid = simulate_weekly_summary(eta_iid)

print("\nBYM weekly")
mean_week, low_week, up_week = simulate_weekly_summary(eta_week)

print("\nBYM weekly + cov")
mean_week_cov, low_week_cov, up_week_cov = simulate_weekly_summary(eta_week_cov)

In [ ]:
# Paths and settings (reuse variables defined above)
N_REP = M
# OBSERVED WEEKLY STATISTIC
week_indices = [(week == w) for w in range(52)]
T_obs_week = np.array([y[:, idx].mean() for idx in week_indices])
# LOAD POSTERIORS (updated filenames)
# IID
iid01 = np.concatenate([
    np.load(BASE_DIR / f"p01_ind_all_chain{c}.npz")["all_theta"][:, ::thin]
    for c in range(n_chains)
], axis=1)

iid10 = np.concatenate([
    np.load(BASE_DIR / f"p10_ind_all_chain{c}.npz")["all_theta"][:, ::thin]
    for c in range(n_chains)
], axis=1)


# BYM weekly
eta01_week, tau01_week, eta10_week, tau10_week = [], [], [], []

for c in range(10):
    with open(BASE_DIR / f"p01_weekly_chain{c}.pkl","rb") as f:
        d = pickle.load(f)
    eta01_week.append(d["eta"][:, ::15])
    tau01_week.append(d["tau"][:, ::15])

    with open(BASE_DIR / f"p10_weekly_chain{c}.pkl","rb") as f:
        d = pickle.load(f)
    eta10_week.append(d["eta"][:, ::15])
    tau10_week.append(d["tau"][:, ::15])

eta01_week = np.concatenate(eta01_week, axis=1)
tau01_week = np.concatenate(tau01_week, axis=1)
eta10_week = np.concatenate(eta10_week, axis=1)
tau10_week = np.concatenate(tau10_week, axis=1)


# BYM weekly + cov
eta01_wf, tau01_wf, eta10_wf, tau10_wf = [], [], [], []

for c in range(10):
    with open(BASE_DIR / f"p01_weekly_cov_chain{c}.pkl","rb") as f:
        d = pickle.load(f)
    eta01_wf.append(d["eta"][:, ::15])
    tau01_wf.append(d["tau"][:, ::15])

    with open(BASE_DIR / f"p10_weekly_cov_chain{c}.pkl","rb") as f:
        d = pickle.load(f)
    eta10_wf.append(d["eta"][:, ::15])
    tau10_wf.append(d["tau"][:, ::15])

eta01_wf = np.concatenate(eta01_wf, axis=1)
tau01_wf = np.concatenate(tau01_wf, axis=1)
eta10_wf = np.concatenate(eta10_wf, axis=1)
tau10_wf = np.concatenate(tau10_wf, axis=1)
# SIGMOID
def sigmoid(z):
    return 1/(1+np.exp(-np.clip(z,-30,30)))
# ETA FUNCTIONS (logic unchanged)
def eta_iid(m,t):
    e1 = sum(cov4[t,k]*iid01[k*S:(k+1)*S,m] for k in range(4))
    e2 = sum(cov4[t,k]*iid10[k*S:(k+1)*S,m] for k in range(4))
    return e1,e2


def eta_week(m,t):
    w = week[t]
    e1 = np.zeros(S)
    e2 = np.zeros(S)
    for k in range(8):
        e1 += cov8[t,k] * eta01_week[k*S:(k+1)*S,m] * tau01_week[k*52+w,m]
        e2 += cov8[t,k] * eta10_week[k*S:(k+1)*S,m] * tau10_week[k*52+w,m]
    return e1,e2


def eta_week_cov(m,t):
    w = week[t]
    e1 = np.zeros(S)
    e2 = np.zeros(S)

    for k in range(8):
        e1 += cov8[t,k] * eta01_wf[k*S:(k+1)*S,m] * tau01_wf[k*52+w,m]
        e2 += cov8[t,k] * eta10_wf[k*S:(k+1)*S,m] * tau10_wf[k*52+w,m]

    gamma1 = eta01_wf[8*S:8*S+3,m]
    gamma2 = eta10_wf[8*S:8*S+3,m]

    fac = np.column_stack([
        t_trend[t]*lat,
        t_trend[t]*elev,
        t_trend[t]*temp[:,t]
    ])

    return e1 + fac @ gamma1, e2 + fac @ gamma2
# PPP (simplified)
def run_ppp(get_eta, name):

    T_pred = np.zeros((M, 52))

    for j, m in enumerate(tqdm(range(M), desc=name)):

        y_rep = np.zeros((S,T), dtype=int)
        y_rep[:,0] = y[:,0]

        for t in range(1,T):
            eta01, eta10 = get_eta(m, t-1)
            p01 = sigmoid(eta01)
            p10 = sigmoid(eta10)
            prev = y_rep[:,t-1]
            y_rep[:,t] = np.random.binomial(1, np.where(prev==0,p01,1-p10))

        for w in range(52):
            T_pred[j,w] = y_rep[:, week_indices[w]].mean()

    return T_pred, (T_pred >= T_obs_week).mean(axis=0)
# RUN
pred_iid, ppp_iid = run_ppp(eta_iid, "IID")
pred_week, ppp_week = run_ppp(eta_week, "BYM weekly")
pred_week_cov, ppp_week_cov = run_ppp(eta_week_cov, "BYM weekly + cov")

In [ ]:
weeks = np.arange(52)
import matplotlib.pyplot as plt
plt.figure(figsize=(11,5))

colors = {
    "IID": "tab:blue",
    "BYM weekly": "tab:orange",
    "BYM weekly + cov": "tab:green"
}

series = {
    "IID": ppp_iid,
    "BYM weekly": ppp_week,
    "BYM weekly + cov": ppp_week_cov
}

# ---- Main curves ----
for name, values in series.items():
    plt.plot(
        weeks,
        values,
        color=colors[name],
        linewidth=1.6,
        marker="o",
        markersize=4,
        alpha=0.9,
        label=name
    )

# ---- Abnormal bands ----
band_height = 0.010
band_positions = {
    "IID": -0.070,
    "BYM weekly": -0.055,
    "BYM weekly + cov": -0.040
}

for name, values in series.items():
    mask = (values < 0.05) | (values > 0.95)
    y0 = band_positions[name]

    for w in weeks[mask]:
        plt.gca().add_patch(
            plt.Rectangle(
                (w - 0.4, y0),
                0.8,
                band_height,
                color=colors[name],
                alpha=0.85
            )
        )

# ---- Left annotation ----
band_min = -0.070
band_max = -0.040 + band_height
band_center = (band_min + band_max) / 2


# ---- Reference lines ----
plt.axhline(0.5, linestyle="--", color="gray", alpha=0.7)
plt.axhline(0.05, linestyle="--", color="red", alpha=0.6)
plt.axhline(0.95, linestyle="--", color="red", alpha=0.6)

plt.ylim(-0.085, 1.05)
plt.yticks(np.arange(0, 1.01, 0.1))

plt.xlabel("Week")
plt.ylabel("Posterior Predictive p-value")
plt.title("Weekly Snowy Cell Proportions Posterior Predictive p-values")

plt.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.22),
    ncol=3,
    frameon=False,
    fontsize=9
)

plt.subplots_adjust(bottom=0.32)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np

weeks = np.arange(1, 53)

df = pd.DataFrame({
    "week": np.tile(weeks, 3),
    "p_value": np.concatenate([ppp_iid, ppp_week, ppp_week_cov]),
    "model": (["IID"] * 52 +
              ["BYM weekly"] * 52 +
              ["BYM weekly + cov"] * 52)
})

df.to_csv("ppp_weekly.csv", index=False)

In [ ]:
# PPC WEEKLY (CLEAN VERSION)
# Models:
#   - IID
#   - BYM weekly
#   - BYM weekly + cov + longitude (NEW)
# Output file: ppp_weekly.csv
import numpy as np
import pyreadr
import pickle
import pandas as pd
from pathlib import Path
from tqdm import tqdm

np.random.seed(1)

BASE_DIR = Path(r"path/to/snow/data-and-results")
PERIOD = 52
thin = 15
n_chains = 10
# Data
snow = pyreadr.read_r(BASE_DIR / "snow_cleaned_full.Rda")
snow = list(snow.values())[0]

coords = snow.iloc[:, :2].to_numpy()
y = snow.iloc[:, 2:].to_numpy()

S, T = y.shape
print("Using FULL S =", S)
# TIME
t_raw = np.arange(1, T+1)
t_trend = (t_raw - t_raw.mean()) / t_raw.std()

cov4 = np.column_stack([
    np.ones(T),
    np.cos(2*np.pi*t_raw/PERIOD),
    np.sin(2*np.pi*t_raw/PERIOD),
    t_trend
])

cov8 = np.column_stack([
    np.ones(T), np.ones(T),
    np.cos(2*np.pi*t_raw/PERIOD), np.cos(2*np.pi*t_raw/PERIOD),
    np.sin(2*np.pi*t_raw/PERIOD), np.sin(2*np.pi*t_raw/PERIOD),
    t_trend, t_trend
])

week = (t_raw - 1) % 52
week_indices = [(week == w) for w in range(52)]
# OBSERVED
T_obs_week = np.array([y[:, idx].mean() for idx in week_indices])
# COVARIATES (FOR +LON MODEL)
lat = (coords[:,1] - coords[:,1].mean()) / coords[:,1].std()

# longitude split
lon_raw = coords[:,0]
region = np.zeros(S)
region[lon_raw >= -30] = 1

lon_na = np.zeros(S)
lon_euas = np.zeros(S)

mask_na = (region == 0)
mask_euas = (region == 1)

lon_na[mask_na] = (
    (lon_raw[mask_na] - lon_raw[mask_na].mean()) /
    max(lon_raw[mask_na].std(), 1e-6)
)

lon_euas[mask_euas] = (
    (lon_raw[mask_euas] - lon_raw[mask_euas].mean()) /
    max(lon_raw[mask_euas].std(), 1e-6)
)

# elevation
no_nbs = np.array([
    57,170,236,269,343,685,946,947,989,
    1037,1084,1090,1109,1118,1127,1176,1203
]) - 1

elev_raw = pd.read_csv(BASE_DIR/"curr_elev.csv").iloc[:,3].to_numpy()
nnbs_elev = pd.read_csv(BASE_DIR/"nnbs_elev.csv", sep="\t").iloc[:,2].to_numpy()

mask = np.ones(S, dtype=bool)
mask[no_nbs] = False

elev_all = np.zeros(S)
elev_all[mask] = elev_raw
elev_all[no_nbs] = nnbs_elev
elev = (elev_all - elev_all.mean()) / elev_all.std()

# temperature
snow_temp = pyreadr.read_r(BASE_DIR/"snow_temp_full.Rda")
snow_temp = list(snow_temp.values())[0]
temp = snow_temp.iloc[:,2:].to_numpy()
temp = (temp - temp.mean()) / temp.std()
# LOAD POSTERIORS
# IID
iid01 = np.concatenate([
    np.load(BASE_DIR / f"p01_ind_all_chain{c}.npz")["all_theta"][:, ::thin]
    for c in range(n_chains)
], axis=1)

iid10 = np.concatenate([
    np.load(BASE_DIR / f"p10_ind_all_chain{c}.npz")["all_theta"][:, ::thin]
    for c in range(n_chains)
], axis=1)

# BYM weekly
def load_bym(prefix):
    eta_list, tau_list = [], []
    for c in range(n_chains):
        with open(BASE_DIR / f"{prefix}_chain{c}.pkl","rb") as f:
            d = pickle.load(f)
        eta_list.append(d["eta"][:, ::thin])
        tau_list.append(d["tau"][:, ::thin])
    return np.concatenate(eta_list,1), np.concatenate(tau_list,1)

eta01_week, tau01_week = load_bym("p01_weekly")
eta10_week, tau10_week = load_bym("p10_weekly")

# BYM with longitude
eta01_wf, tau01_wf = load_bym("p01_weekly_cov+lon")
eta10_wf, tau10_wf = load_bym("p10_weekly_cov+lon")

M = iid01.shape[1]
# SIGMOID
def sigmoid(z):
    return 1/(1+np.exp(-np.clip(z,-30,30)))
# ETA FUNCTIONS
# -------- IID (unchanged)
def eta_iid(m,t):
    e1 = sum(cov4[t,k]*iid01[k*S:(k+1)*S,m] for k in range(4))
    e2 = sum(cov4[t,k]*iid10[k*S:(k+1)*S,m] for k in range(4))
    return e1,e2


# -------- BYM weekly (unchanged)
def eta_week(m,t):
    w = week[t]
    e1 = np.zeros(S)
    e2 = np.zeros(S)
    for k in range(8):
        e1 += cov8[t,k]*eta01_week[k*S:(k+1)*S,m]*tau01_week[k*52+w,m]
        e2 += cov8[t,k]*eta10_week[k*S:(k+1)*S,m]*tau10_week[k*52+w,m]
    return e1,e2


# -------- BYM + covariates + longitude (only modified section)
def eta_week_cov_lon(m,t):

    w = week[t]
    e1 = np.zeros(S)
    e2 = np.zeros(S)

    for k in range(8):
        e1 += cov8[t,k]*eta01_wf[k*S:(k+1)*S,m]*tau01_wf[k*52+w,m]
        e2 += cov8[t,k]*eta10_wf[k*S:(k+1)*S,m]*tau10_wf[k*52+w,m]

    gamma1 = eta01_wf[8*S:8*S+5,m]
    gamma2 = eta10_wf[8*S:8*S+5,m]

    fac = np.column_stack([
        t_trend[t]*lon_na,
        t_trend[t]*lon_euas,
        t_trend[t]*lat,
        t_trend[t]*elev,
        t_trend[t]*temp[:,t]
    ])

    return e1 + fac @ gamma1, e2 + fac @ gamma2
# SIMULATOR
def run_ppp(get_eta, name):

    T_pred = np.zeros((M, 52))

    for j, m in enumerate(tqdm(range(M), desc=name)):

        y_rep = np.zeros((S,T), dtype=int)
        y_rep[:,0] = y[:,0]

        for t in range(1,T):
            eta01, eta10 = get_eta(m, t-1)
            p01 = sigmoid(eta01)
            p10 = sigmoid(eta10)
            prev = y_rep[:,t-1]
            y_rep[:,t] = np.random.binomial(1, np.where(prev==0,p01,1-p10))

        for w in range(52):
            T_pred[j,w] = y_rep[:, week_indices[w]].mean()

    return (T_pred >= T_obs_week).mean(axis=0)
# RUN
ppp_iid = run_ppp(eta_iid, "IID")
ppp_week = run_ppp(eta_week, "BYM weekly")
ppp_week_cov = run_ppp(eta_week_cov_lon, "BYM weekly + cov + lon")
# SAVE CSV
weeks = np.arange(1,53)

df = pd.DataFrame({
    "week": np.tile(weeks, 3),
    "p_value": np.concatenate([ppp_iid, ppp_week, ppp_week_cov]),
    "model": (["IID"]*52 +
              ["BYM weekly"]*52 +
              ["BYM weekly + cov + lon"]*52)
})

df.to_csv("ppp_weekly.csv", index=False)

print("\nSaved to ppp_weekly.csv")